# Task 2g: Model Inference within CMSSW (Conceptual + ONNX Export)
## Introduction
This task focuses on deploying the trained deep learning model within the CMS Software Framework (CMSSW) and evaluating inference performance.

Due to environment and system constraints, full CMSSW-based inference could not be executed locally. However, the necessary steps for deployment, including model conversion to ONNX format and integration into CMSSW, are outlined and discussed.


---

## Model Conversion to ONNX

To enable compatibility with CMSSW, the trained PyTorch model is converted to ONNX format. ONNX (Open Neural Network Exchange) provides a standardized format for deploying machine learning models across different frameworks.

**The Fully Connected model was selected for deployment as it demonstrated better learning performance compared to the CNN model.**

In [4]:
import torch
import torch.nn as nn

# ===== Define SimpleModel (FC model) =====
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4*125*125, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze()


# ===== Load trained model =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleModel()
model.load_state_dict(torch.load("models/best_model_fc.pth", map_location=device))  
model.to(device)
model.eval()

# ===== Dummy input =====
dummy_input = torch.randn(1, 4, 125, 125).to(device)

# ===== Export to ONNX =====
torch.onnx.export(
    model,
    dummy_input,
    "best_model_fc.onnx",   # ✅ final ONNX file
    input_names=["input"],
    output_names=["output"],
    opset_version=11
)

print("✅ Fully Connected model exported to ONNX format")

W0327 11:15:56.225000 1484 site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


C:\Users\LENOVO\anaconda3\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\LENOVO\anaconda3\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Users\LENOVO\anaconda3\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Users\LENOVO\anaconda3\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, 

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
✅ Fully Connected model exported to ONNX format



---

## CMSSW Setup (Conceptual)

The CMSSW framework can be set up using Docker images provided by CERN. The following steps are typically used:

1. Pull the CMSSW Docker image
2. Initialize CMSSW environment
3. Compile necessary packages
4. Integrate the ONNX model into the inference pipeline

Example commands:

cmsrel CMSSW_11_0_1  
cd CMSSW_11_0_1/src/  
cmsenv  

scram b -j8

## Inference in CMSSW

Once the model is converted to ONNX format, it can be integrated into the CMSSW inference pipeline.

Inference can be executed using:

cmsRun RecoE2E/EGTagger/python/EGInference_cfg.py \
inputFiles=file:SIM_DoubleGammaPt50_Pythia8_1000Ev.root \
maxEvents=-1 \
EGModelName=model.onnx

This step enables evaluating the model within a real high-energy physics workflow.

## Challenges and Limitations

Due to system and environment constraints, full CMSSW-based inference could not be executed locally. Setting up CMSSW requires specific dependencies, Docker configuration, and access to CERN software repositories.

However, the model was successfully converted to ONNX format, which is the key requirement for deployment within CMSSW.

## Conclusion

This task demonstrates the process of preparing a trained deep learning model for deployment in a real experimental framework. While full CMSSW execution was not performed, the ONNX conversion and pipeline understanding highlight the practical challenges of deploying machine learning models in high-energy physics environments.